# Kite — Kaggle model team

This notebook prepares the model roster from the Kite design. The AWS router chooses a role/model in the task payload; the notebook loads only that selected model for the run. Each request is manual/batch for now: the live Kaggle adapter is not yet connected to AWS.

Do not put private code, credentials, or tokens in prompt cells. Notebook source and outputs may persist.

## Kaggle setup

1. Import this notebook into Kaggle and keep it private.
2. Select GPU T4 x2 if Kaggle offers it; enable Internet to download packages and model weights.
3. Run the install and GPU check cells once.
4. Set role/model_id in TASK to match the router decision, then run the remaining cells.
5. Only one model is loaded at a time. To switch models, change TASK.model_id, rerun the model-load cell, then run inference/export.
6. Stop the Kaggle session when finished.

## Router roster

| Role | Default model |
|---|---|
| Architect / hard debugging | deepseek-ai/DeepSeek-R1-Distill-Qwen-32B |
| Automatic file editing | mistralai/Devstral-Small-2-24B-Instruct-2512 |
| Coding / review | Qwen/Qwen2.5-Coder-32B-Instruct |
| Image-based UI work | Qwen/Qwen2.5-VL-7B-Instruct; 32B variant also selectable |
| Fast code | google/gemma-4-26B-A4B-it |
| File search | BAAI/bge-m3 or nomic-ai/nomic-embed-text-v1.5 |

The selected model_id can be supplied by the AWS coordinator once a supported job transport exists. Today the task cell is the manual input point.

In [ ]:
%pip -q install -U git+https://github.com/huggingface/transformers.git accelerate bitsandbytes safetensors sentence-transformers qwen-vl-utils

In [ ]:
import gc
import json
import shutil
import time
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU is enabled. Choose an available GPU in Kaggle Notebook settings.")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
free_gb = round(shutil.disk_usage("/kaggle/working").free / (1024 ** 3), 1)
print(f"Free notebook disk: {free_gb} GiB")

In [ ]:
# Models are downloaded on demand; 4-bit loading reduces VRAM, not raw checkpoint download size.
MODEL_CATALOG = {
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B": {
        "family": "causal", "approx_download_gb": 65,
    },
    "mistralai/Devstral-Small-2-24B-Instruct-2512": {
        "family": "mistral3", "approx_download_gb": 48,
    },
    "Qwen/Qwen2.5-Coder-32B-Instruct": {
        "family": "causal", "approx_download_gb": 63,
    },
    "Qwen/Qwen2.5-VL-7B-Instruct": {
        "family": "qwen_vl", "approx_download_gb": 16,
    },
    "Qwen/Qwen2.5-VL-32B-Instruct": {
        "family": "qwen_vl", "approx_download_gb": 63,
    },
    "google/gemma-4-26B-A4B-it": {
        "family": "gemma4", "approx_download_gb": 52,
    },
    "BAAI/bge-m3": {
        "family": "embedding", "approx_download_gb": 2.5,
    },
    "nomic-ai/nomic-embed-text-v1.5": {
        "family": "embedding", "approx_download_gb": 0.6,
    },
}

ROLE_DEFAULTS = {
    "architect": "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B",
    "file_editor": "mistralai/Devstral-Small-2-24B-Instruct-2512",
    "coder_review": "Qwen/Qwen2.5-Coder-32B-Instruct",
    "frontend_vision": "Qwen/Qwen2.5-VL-7B-Instruct",
    "fast_code": "google/gemma-4-26B-A4B-it",
    "file_search": "BAAI/bge-m3",
}

# The AWS coordinator's task payload will supply model_id after the provider transport is implemented.
TASK = {
    "task_id": "manual-001",
    "role": "coder_review",
    "model_id": "Qwen/Qwen2.5-Coder-32B-Instruct",
    "system_prompt": "You are a careful coding assistant. Give a concise, correct answer.",
    "prompt": "Explain one safe way to validate a Python function that removes duplicate items while preserving order.",
    "image_path": None,  # Vision request: use a local Kaggle input path such as /kaggle/input/my-data/screenshot.png
    "embedding_task": "search_query",  # Nomic prefix used for embedding tasks.
    "max_new_tokens": 512,
}

MODEL_ID = TASK.get("model_id") or ROLE_DEFAULTS[TASK["role"]]
if MODEL_ID not in MODEL_CATALOG:
    raise ValueError(f"Unsupported model_id: {MODEL_ID}. Add it to MODEL_CATALOG first.")
MODEL_SPEC = MODEL_CATALOG[MODEL_ID]
print("Role:", TASK["role"])
print("Selected model:", MODEL_ID, "| family:", MODEL_SPEC["family"])
print("Approx. checkpoint download:", MODEL_SPEC["approx_download_gb"], "GiB")

## Load the selected model

The model is loaded only after TASK.model_id is selected. 4-bit NF4 loading is used for the language/vision models to reduce GPU memory, but the full source checkpoint still needs to download and occupy disk. If the disk preflight fails, select the Qwen2.5-VL 7B or an embedding model. A larger checkpoint can still run out of Kaggle disk or memory; the 2×T4 allocation is not guaranteed.

In [ ]:
from transformers import BitsAndBytesConfig

if free_gb < MODEL_SPEC["approx_download_gb"] + 3:
    raise RuntimeError(
        f"Not enough free notebook disk for {MODEL_ID}: about "
        f"{MODEL_SPEC['approx_download_gb']} GiB needed plus working space; "
        f"{free_gb} GiB free. Choose a smaller model or free notebook disk."
    )

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
load_kwargs = {
    "device_map": "auto",
    "torch_dtype": torch.float16,
    "quantization_config": quantization,
}
family = MODEL_SPEC["family"]
model = tokenizer = processor = embedding_model = None

if family == "causal":
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
elif family == "qwen_vl":
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, **load_kwargs)
elif family == "gemma4":
    from transformers import AutoModelForMultimodalLM, AutoProcessor
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, **load_kwargs)
elif family == "mistral3":
    from transformers import Mistral3ForConditionalGeneration, MistralCommonBackend
    tokenizer = MistralCommonBackend.from_pretrained(MODEL_ID)
    model = Mistral3ForConditionalGeneration.from_pretrained(MODEL_ID, **load_kwargs)
elif family == "embedding":
    from sentence_transformers import SentenceTransformer
    embedding_model = SentenceTransformer(
        MODEL_ID,
        device="cuda",
        trust_remote_code=(MODEL_ID == "nomic-ai/nomic-embed-text-v1.5"),
    )
else:
    raise ValueError(f"Unsupported model family: {family}")

if model is not None:
    model.eval()
print("Loaded selected model:", MODEL_ID)

## Run the routed task

For a vision role, attach a Kaggle dataset containing the image and set TASK.image_path. The embedding roles export vectors instead of generated text. Only one model is resident at a time; switching model_id reloads model weights.

In [ ]:
started = time.monotonic()
result_text = None
embedding_values = None
input_token_count = 0
output_token_count = 0

if family == "embedding":
    text = TASK["prompt"]
    if MODEL_ID == "nomic-ai/nomic-embed-text-v1.5":
        text = f"{TASK.get('embedding_task', 'search_query')}: {text}"
    vector = embedding_model.encode(
        [text], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False
    )[0]
    embedding_values = vector.tolist()
    result_text = f"Embedding generated; dimension={len(embedding_values)}"
else:
    if family == "mistral3":
        messages = [
            {"role": "system", "content": TASK["system_prompt"]},
            {"role": "user", "content": [{"type": "text", "text": TASK["prompt"]}]},
        ]
        inputs = tokenizer.apply_chat_template(
            conversation=messages, return_tensors="pt", return_dict=True
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    elif family in ("qwen_vl", "gemma4"):
        from PIL import Image
        content = []
        image_path = TASK.get("image_path")
        if image_path:
            if not Path(image_path).is_file():
                raise FileNotFoundError(f"Image not found: {image_path}")
            content.append({"type": "image", "image": Image.open(image_path).convert("RGB")})
        content.append({"type": "text", "text": TASK["prompt"]})
        messages = [
            {"role": "system", "content": TASK["system_prompt"]},
            {"role": "user", "content": content},
        ]
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
        inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}
    else:
        messages = [
            {"role": "system", "content": TASK["system_prompt"]},
            {"role": "user", "content": TASK["prompt"]},
        ]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    input_token_count = int(inputs["input_ids"].shape[-1])
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs, max_new_tokens=int(TASK.get("max_new_tokens", 512)), do_sample=False
        )
    new_ids = output_ids[0, input_token_count:]
    output_token_count = int(new_ids.numel())
    if family in ("qwen_vl", "gemma4"):
        result_text = processor.decode(new_ids, skip_special_tokens=True).strip()
    else:
        result_text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()

elapsed_seconds = round(time.monotonic() - started, 3)
print(result_text)

## Export result

This is a manual batch response envelope, not Kite's leased WorkerResult. It identifies the selected model and role; it does not edit a repository or push code.

In [ ]:
response = {
    "schema_version": 1,
    "task_id": TASK["task_id"],
    "role": TASK["role"],
    "model_id": MODEL_ID,
    "status": "completed",
    "text": result_text,
    "usage": {"input_tokens": input_token_count, "output_tokens": output_token_count},
    "elapsed_seconds": elapsed_seconds,
}
if embedding_values is not None:
    response["embedding"] = embedding_values

OUTPUT_PATH = "/kaggle/working/kite_worker_result.json"
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(response, f, ensure_ascii=False)
print("Saved:", OUTPUT_PATH)
print("Result model:", response["model_id"])

## Stop cleanly

Download any needed result, then stop the Kaggle notebook session to release its accelerator.

In [ ]:
del model, tokenizer, processor, embedding_model
gc.collect()
torch.cuda.empty_cache()
print("Model memory released. Stop the Kaggle session from the notebook controls.")